# Cleaning_CYRV_geolocation_dataset.csv

## Conclusion

The cleaned geolocation dataset contains 738,299 records and preserves the original five-column structure.

Only records with strong evidence of data-quality problems were removed or corrected. Geographic and naming variations that could represent legitimate locations or source-system differences were retained rather than standardized aggressively.

The dataset is now ready to be saved and used as a cleaned geographic reference for further analysis and validation of customer and seller location data.

## Cleaning Summary

The geolocation dataset was inspected and cleaned to improve consistency and reliability while preserving valid geographic information.

### Cleaning actions performed

- Removed 261,831 fully duplicated rows.
- Standardized whitespace in `geolocation_city` by removing leading/trailing spaces and replacing multiple consecutive spaces with a single space.
- Identified and removed 25 invalid geographic coordinate records after investigating coordinate outliers and comparing affected ZIP codes with other records and external geographic references.
- Investigated ZIP code prefixes associated with multiple states.
- Removed 8 confirmed incorrect ZIP/state records after internal comparison and external validation.
- Corrected 2 encoded apostrophe artifacts (`%26apos%3b`) in city names.
- Retained legitimate city spelling variations, accented/unaccented forms, and locality variations where there was insufficient evidence that they represented data errors.
- Retained `geolocation_zip_code_prefix` as `int64` for consistency with the customer and seller datasets.

### Final validation

- Final shape: 738,299 rows × 5 columns
- Missing values: 0
- Fully duplicated rows: 0
- Invalid latitude values: 0
- Invalid longitude values: 0
- Remaining investigated coordinate outliers: 0
- ZIP code prefixes associated with multiple states: 0
- City spacing issues: 0
- Encoded city artifacts: 0

In [1]:
import pandas as pd

geolocation = pd.read_csv("/Users/yuliiapotrymai/Desktop/SpikupCapstone2026_CYRV/data/cyrv/CYRV_geolocation_dataset.csv")

geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [2]:
print("Dataset shape:", geolocation.shape)

print("\nColumns:")
print(geolocation.columns.tolist())

print("\nData types:")
print(geolocation.dtypes)

Dataset shape: (1000163, 5)

Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object


In [3]:
missing_values = geolocation.isna().sum()

missing_percent = (
    geolocation.isna().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percent": missing_percent
})

missing_summary

,missing_count,missing_percent
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


In [4]:
print(
    "Duplicate rows:",
    geolocation.duplicated().sum()
)

print(
    "Duplicate percentage:",
    round(
        geolocation.duplicated().mean() * 100,
        2
    ),
    "%"
)


Duplicate rows: 261831
Duplicate percentage: 26.18 %


In [5]:
duplicate_rows = geolocation[
    geolocation.duplicated(keep=False)
].sort_values(
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
)

print("Rows involved in duplicate groups:", len(duplicate_rows))

duplicate_rows.head(20)

Rows involved in duplicate groups: 390005


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.551337,-46.634027,sao paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP


In [6]:
geolocation_deduplicated = geolocation.drop_duplicates()

print("Original rows:", len(geolocation))
print("Duplicate rows to remove:", geolocation.duplicated().sum())
print("Rows after removing duplicates:", len(geolocation_deduplicated))
print(
    "Duplicate rows remaining:",
    geolocation_deduplicated.duplicated().sum()
)

Original rows: 1000163
Duplicate rows to remove: 261831
Rows after removing duplicates: 738332
Duplicate rows remaining: 0


In [7]:
geolocation = geolocation.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", geolocation.shape)
print("Duplicate rows:", geolocation.duplicated().sum())

Shape after removing duplicates: (738332, 5)
Duplicate rows: 0


In [8]:
print(
    "Min ZIP prefix:",
    geolocation["geolocation_zip_code_prefix"].min()
)

print(
    "Max ZIP prefix:",
    geolocation["geolocation_zip_code_prefix"].max()
)

print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

print("\nZIP prefix digit lengths:")
print(
    geolocation["geolocation_zip_code_prefix"]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

Min ZIP prefix: 1001
Max ZIP prefix: 99990
Unique ZIP prefixes: 19015

ZIP prefix digit lengths:
geolocation_zip_code_prefix
4    158832
5    579500
Name: count, dtype: int64


In [9]:
print(
    "Unique states:",
    geolocation["geolocation_state"].nunique()
)

print("\nState values:")
print(
    geolocation["geolocation_state"]
    .value_counts()
    .sort_index()
)

print(
    "\nInvalid state length:",
    geolocation["geolocation_state"]
    .str.len()
    .ne(2)
    .sum()
)

print(
    "Leading/trailing spaces:",
    geolocation["geolocation_state"]
    .ne(geolocation["geolocation_state"].str.strip())
    .sum()
)

print(
    "Contains lowercase:",
    geolocation["geolocation_state"]
    .str.contains(r"[a-z]", regex=True)
    .sum()
)

print(
    "Empty/whitespace-only:",
    geolocation["geolocation_state"]
    .str.strip()
    .eq("")
    .sum()
)

Unique states: 27

State values:
geolocation_state
AC      1039
AL      3415
AM      1986
AP       738
BA     27720
CE      9541
DF      9080
ES     12632
GO     15601
MA      6277
MG    101353
MS      8594
MT      9374
PA      8551
PB      4787
PE     13162
PI      3592
PR     45059
RJ     78836
RN      4014
RO      2523
RR       568
RS     48093
SC     30191
SE      2653
SP    285976
TO      2977
Name: count, dtype: int64

Invalid state length: 0
Leading/trailing spaces: 0
Contains lowercase: 0
Empty/whitespace-only: 0


In [10]:
print(
    "Unique cities:",
    geolocation["geolocation_city"].nunique()
)

print(
    "Leading/trailing spaces:",
    geolocation["geolocation_city"]
    .ne(geolocation["geolocation_city"].str.strip())
    .sum()
)

print(
    "Contains uppercase:",
    geolocation["geolocation_city"]
    .str.contains(r"[A-Z]", regex=True)
    .sum()
)

print(
    "Empty/whitespace-only:",
    geolocation["geolocation_city"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Contains multiple spaces:",
    geolocation["geolocation_city"]
    .str.contains(r"\s{2,}", regex=True)
    .sum()
)

Unique cities: 8011
Leading/trailing spaces: 1
Contains uppercase: 0
Empty/whitespace-only: 0
Contains multiple spaces: 11


In [11]:
spacing_issues = geolocation[
    geolocation["geolocation_city"].ne(
        geolocation["geolocation_city"].str.strip()
    )
    |
    geolocation["geolocation_city"].str.contains(
        r"\s{2,}",
        regex=True
    )
][
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ]
]

print("Rows with spacing issues:", len(spacing_issues))

spacing_issues.sort_values(
    [
        "geolocation_state",
        "geolocation_city"
    ]
)

Rows with spacing issues: 12


,geolocation_zip_code_prefix,geolocation_city,geolocation_state
525726,57925,barra de santo antônio,AL
479797,40243,salvador,BA
560916,68545,pau d arco,PA
641355,85896,diamante d oeste,PR
642571,85896,diamante d oeste,PR
638954,85580,itapejara d oeste,PR
639318,85580,itapejara d oeste,PR
639429,85580,itapejara d oeste,PR
639703,85580,itapejara d oeste,PR
639784,85580,itapejara d oeste,PR


In [12]:
geolocation["geolocation_city"] = (
    geolocation["geolocation_city"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print(
    "Leading/trailing spaces:",
    geolocation["geolocation_city"]
    .ne(geolocation["geolocation_city"].str.strip())
    .sum()
)

print(
    "Contains multiple spaces:",
    geolocation["geolocation_city"]
    .str.contains(r"\s{2,}", regex=True)
    .sum()
)

print(
    "Unique cities after spacing cleanup:",
    geolocation["geolocation_city"].nunique()
)

Leading/trailing spaces: 0
Contains multiple spaces: 0
Unique cities after spacing cleanup: 8009


In [13]:
print("Latitude:")
print("Min:", geolocation["geolocation_lat"].min())
print("Max:", geolocation["geolocation_lat"].max())

print("\nLongitude:")
print("Min:", geolocation["geolocation_lng"].min())
print("Max:", geolocation["geolocation_lng"].max())

print(
    "\nInvalid latitude values:",
    (~geolocation["geolocation_lat"].between(-90, 90)).sum()
)

print(
    "Invalid longitude values:",
    (~geolocation["geolocation_lng"].between(-180, 180)).sum()
)

Latitude:
Min: -36.6053744107061
Max: 45.06593318269697

Longitude:
Min: -101.46676644931476
Max: 121.10539381057764

Invalid latitude values: 0
Invalid longitude values: 0


In [14]:
extreme_coordinates = pd.concat([
    geolocation.nsmallest(10, "geolocation_lat"),
    geolocation.nlargest(10, "geolocation_lat"),
    geolocation.nsmallest(10, "geolocation_lng"),
    geolocation.nlargest(10, "geolocation_lng")
]).drop_duplicates()

extreme_coordinates[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values(
    ["geolocation_lat", "geolocation_lng"]
)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
732029,98780,-36.605374,-64.283946,santa rosa,RS
732479,98780,-36.603837,-64.287433,santa rosa,RS
499010,45936,-34.622400,-58.901888,itabata,BA
355603,28155,-34.586422,-58.732101,santa maria,RJ
720850,96255,-33.692616,-53.453972,chui,RS
720867,96255,-33.692504,-53.456158,chui,RS
720854,96255,-33.692491,-53.456402,chui,RS
720594,96255,-33.692291,-53.460274,chui,RS
721401,96255,-33.692255,-53.460853,chui,RS
721444,96255,-33.692196,-53.456586,chuí,RS


In [15]:
coordinate_outliers = geolocation[
    ~(
        geolocation["geolocation_lat"].between(-35, 6)
        &
        geolocation["geolocation_lng"].between(-75, -30)
    )
]

print("Potential coordinate outliers:", len(coordinate_outliers))

print(
    "Percentage:",
    round(len(coordinate_outliers) / len(geolocation) * 100, 4),
    "%"
)

print(
    "Unique ZIP prefixes affected:",
    coordinate_outliers["geolocation_zip_code_prefix"].nunique()
)

Potential coordinate outliers: 25
Percentage: 0.0034 %
Unique ZIP prefixes affected: 20


In [16]:
outlier_zips = coordinate_outliers[
    "geolocation_zip_code_prefix"
].unique()

zip_coordinate_summary = (
    geolocation[
        geolocation["geolocation_zip_code_prefix"].isin(outlier_zips)
    ]
    .assign(
        potential_outlier=lambda df: ~(
            df["geolocation_lat"].between(-35, 6)
            &
            df["geolocation_lng"].between(-75, -30)
        )
    )
    .groupby("geolocation_zip_code_prefix")
    .agg(
        total_rows=("geolocation_zip_code_prefix", "size"),
        outlier_rows=("potential_outlier", "sum"),
        min_lat=("geolocation_lat", "min"),
        max_lat=("geolocation_lat", "max"),
        min_lng=("geolocation_lng", "min"),
        max_lng=("geolocation_lng", "max")
    )
    .sort_values("outlier_rows", ascending=False)
)

zip_coordinate_summary


,total_rows,outlier_rows,min_lat,max_lat,min_lng,max_lng
geolocation_zip_code_prefix,,,,,,
68275,6,3,-1.743515,42.167251,-56.378018,-6.898531
98780,120,2,-36.605374,-7.232281,-64.287433,-35.904965
29654,4,2,-19.825000,29.409252,-101.466766,-40.654440
35179,67,2,-30.048103,25.995245,-98.078544,-42.500182
58441,4,1,-7.276127,41.385328,-36.916056,-8.717342
95130,1,1,14.585073,14.585073,121.105394,121.105394
83810,6,1,-25.869440,39.057629,-50.436930,-9.400037
83252,1,1,42.184003,42.184003,-8.723762,-8.723762
78131,1,1,38.816816,38.816816,-9.394625,-9.394625


In [17]:
coordinate_outliers[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values(
    [
        "geolocation_state",
        "geolocation_zip_code_prefix"
    ]
)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
524889,57319,45.065933,9.341528,pau d'arco,AL
499012,45936,38.323939,-6.775035,itabatan,BA
500424,46560,38.991963,-4.947823,ibiajara,BA
501576,47310,38.268205,-7.803886,santana do sobrado,BA
374435,29654,29.409252,-98.484121,santo antônio do canaã,ES
374467,29654,21.657547,-101.466766,santo antonio do canaa,ES
409379,35179,25.995203,-98.078544,santana do paraíso,MG
409397,35179,25.995245,-98.078533,santana do paraiso,MG
599205,78131,38.816816,-9.394625,varzea grande,MT
558526,68275,41.146203,-8.577855,porto trombetas,PA


In [18]:
outlier_comparison = []

for _, row in coordinate_outliers.iterrows():
    same_location = geolocation[
        (geolocation["geolocation_zip_code_prefix"] == row["geolocation_zip_code_prefix"])
        & (geolocation["geolocation_city"] == row["geolocation_city"])
        & (geolocation["geolocation_state"] == row["geolocation_state"])
    ]

    normal_coordinates = same_location[
        same_location["geolocation_lat"].between(-35, 6)
        & same_location["geolocation_lng"].between(-75, -30)
    ]

    outlier_comparison.append({
        "zip": row["geolocation_zip_code_prefix"],
        "city": row["geolocation_city"],
        "state": row["geolocation_state"],
        "outlier_lat": row["geolocation_lat"],
        "outlier_lng": row["geolocation_lng"],
        "normal_matches": len(normal_coordinates)
    })

outlier_comparison = pd.DataFrame(outlier_comparison)

outlier_comparison.sort_values(
    ["normal_matches", "state", "zip"],
    ascending=[False, True, True]
)

,zip,city,state,outlier_lat,outlier_lng,normal_matches
23,98780,santa rosa,RS,-36.605374,-64.283946,118
24,98780,santa rosa,RS,-36.603837,-64.287433,118
8,35179,santana do paraiso,MG,25.995245,-98.078533,57
9,45936,itabatan,BA,38.323939,-6.775035,12
7,35179,santana do paraíso,MG,25.995203,-98.078544,8
4,28595,portela,RJ,43.684961,-7.411080,6
21,83810,areia branca dos assis,PR,39.057629,-9.400037,4
2,28155,santa maria,RJ,42.439286,13.820214,4
3,28333,raposo,RJ,38.381672,-6.328200,4
11,47310,santana do sobrado,BA,38.268205,-7.803886,3


In [19]:
zero_match_zips = outlier_comparison.loc[
    outlier_comparison["normal_matches"] == 0,
    "zip"
].unique()

geolocation[
    geolocation["geolocation_zip_code_prefix"].isin(zero_match_zips)
][
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values(
    [
        "geolocation_zip_code_prefix",
        "geolocation_city"
    ]
)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
272046,18243,28.008978,-15.536867,bom retiro da esperanca,SP
374343,29654,-19.823680,-40.655474,santo antonio do canaa,ES
374467,29654,21.657547,-101.466766,santo antonio do canaa,ES
374486,29654,-19.825000,-40.654440,santo antonio do canaa,ES
374435,29654,29.409252,-98.484121,santo antônio do canaã,ES
524889,57319,45.065933,9.341528,pau d'arco,AL
524738,57319,-23.258224,-47.307430,vila sao francisco,AL
599205,78131,38.816816,-9.394625,varzea grande,MT
627844,83252,42.184003,-8.723762,ilha dos valadares,PR
710355,95130,14.585073,121.105394,santa lucia do piai,RS


In [20]:
geolocation = geolocation.drop(
    index=coordinate_outliers.index
).reset_index(drop=True)

print("Shape after removing coordinate outliers:", geolocation.shape)

remaining_outliers = geolocation[
    ~(
        geolocation["geolocation_lat"].between(-35, 6)
        &
        geolocation["geolocation_lng"].between(-75, -30)
    )
]

print("Remaining coordinate outliers:", len(remaining_outliers))

Shape after removing coordinate outliers: (738307, 5)
Remaining coordinate outliers: 0


In [21]:
zip_state_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

print(
    "ZIP prefixes with multiple states:",
    (zip_state_counts > 1).sum()
)

print(
    "Maximum states for one ZIP prefix:",
    zip_state_counts.max()
)

ZIP prefixes with multiple states: 8
Maximum states for one ZIP prefix: 2


In [22]:
multi_state_zips = zip_state_counts[
    zip_state_counts > 1
].index

zip_state_conflicts = (
    geolocation[
        geolocation["geolocation_zip_code_prefix"].isin(
            multi_state_zips
        )
    ][
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "geolocation_zip_code_prefix",
            "geolocation_state",
            "geolocation_city"
        ]
    )
)

zip_state_conflicts

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
11127,2116,são paulo,RN
10753,2116,sao paulo,SP
11085,2116,são paulo,SP
45154,4011,sao paulo,AC
43812,4011,sao paulo,SP
44008,4011,são paulo,SP
301132,21550,rio de janeiro,AC
300585,21550,rio de janeiro,RJ
316881,23056,rio de janeiro,AC
315879,23056,rio de janeiro,RJ


In [23]:
zip_state_conflict_counts = (
    geolocation[
        geolocation["geolocation_zip_code_prefix"].isin(
            multi_state_zips
        )
    ]
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ]
    )
    .size()
    .reset_index(name="row_count")
    .sort_values(
        [
            "geolocation_zip_code_prefix",
            "row_count"
        ],
        ascending=[True, False]
    )
)

zip_state_conflict_counts

,geolocation_zip_code_prefix,geolocation_state,row_count
1,2116,SP,10
0,2116,RN,1
3,4011,SP,69
2,4011,AC,1
5,21550,RJ,144
4,21550,AC,1
7,23056,RJ,30
6,23056,AC,1
9,72915,GO,9
8,72915,DF,1


In [24]:
single_state_conflicts = (
    zip_state_conflict_counts[
        zip_state_conflict_counts["row_count"] == 1
    ][
        [
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ]
    ]
)

single_state_conflict_rows = geolocation.merge(
    single_state_conflicts,
    on=[
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ],
    how="inner"
)

single_state_conflict_rows[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values("geolocation_zip_code_prefix")

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,2116,-23.515978,-46.582170,são paulo,RN
1,4011,-23.578707,-46.645779,sao paulo,AC
2,21550,-22.857861,-43.352613,rio de janeiro,AC
3,23056,-22.919164,-43.611097,rio de janeiro,AC
4,72915,-12.408760,-46.428287,taguatinga,DF
5,78557,-11.875142,-55.503020,nova brasilandia d'oeste,RO
6,79750,-22.242062,-53.343159,nova andradina,RS
7,80630,-26.757371,-48.675738,balneario de picarras,SC


In [25]:
conflict_location_counts = (
    geolocation[
        geolocation["geolocation_zip_code_prefix"].isin(
            multi_state_zips
        )
    ]
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    )
    .size()
    .reset_index(name="row_count")
    .sort_values(
        [
            "geolocation_zip_code_prefix",
            "row_count"
        ],
        ascending=[True, False]
    )
)

conflict_location_counts

,geolocation_zip_code_prefix,geolocation_city,geolocation_state,row_count
0,2116,sao paulo,SP,7
2,2116,são paulo,SP,3
1,2116,são paulo,RN,1
4,4011,sao paulo,SP,51
5,4011,são paulo,SP,18
3,4011,sao paulo,AC,1
7,21550,rio de janeiro,RJ,144
6,21550,rio de janeiro,AC,1
9,23056,rio de janeiro,RJ,30
8,23056,rio de janeiro,AC,1


In [26]:
print("Shape:", geolocation.shape)

zip_state_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

print(
    "ZIP prefixes with multiple states:",
    (zip_state_counts > 1).sum()
)

Shape: (738307, 5)
ZIP prefixes with multiple states: 8


In [27]:
invalid_zip_state_pairs = {
    (2116, "RN"),
    (4011, "AC"),
    (21550, "AC"),
    (23056, "AC"),
    (72915, "DF"),
    (78557, "RO"),
    (79750, "RS"),
    (80630, "SC")
}

invalid_mask = geolocation.apply(
    lambda row: (
        row["geolocation_zip_code_prefix"],
        row["geolocation_state"]
    ) in invalid_zip_state_pairs,
    axis=1
)

print("Rows to remove:", invalid_mask.sum())

Rows to remove: 8


In [28]:
geolocation = geolocation[
    ~invalid_mask
].reset_index(drop=True)

zip_state_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

print("Shape:", geolocation.shape)

print(
    "ZIP prefixes with multiple states:",
    (zip_state_counts > 1).sum()
)

Shape: (738299, 5)
ZIP prefixes with multiple states: 0


In [29]:
zip_city_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["geolocation_city"]
    .nunique()
)

print(
    "ZIP prefixes with multiple cities:",
    (zip_city_counts > 1).sum()
)

print(
    "Maximum cities for one ZIP prefix:",
    zip_city_counts.max()
)

ZIP prefixes with multiple cities: 8551
Maximum cities for one ZIP prefix: 5


In [30]:
print(
    zip_city_counts
    .value_counts()
    .sort_index()
)

geolocation_city
1    10460
2     8262
3      253
4       27
5        9
Name: count, dtype: int64


In [31]:
import unicodedata

def normalize_city(value):
    return "".join(
        char
        for char in unicodedata.normalize("NFKD", value)
        if not unicodedata.combining(char)
    ).strip().lower()

geolocation["city_normalized"] = (
    geolocation["geolocation_city"]
    .apply(normalize_city)
)

normalized_zip_city_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["city_normalized"]
    .nunique()
)

print(
    "ZIP prefixes with multiple normalized cities:",
    (normalized_zip_city_counts > 1).sum()
)

print(
    "Maximum normalized cities for one ZIP prefix:",
    normalized_zip_city_counts.max()
)

ZIP prefixes with multiple normalized cities: 544
Maximum normalized cities for one ZIP prefix: 4


In [32]:
print(
    normalized_zip_city_counts
    .value_counts()
    .sort_index()
)

city_normalized
1    18467
2      502
3       37
4        5
Name: count, dtype: int64


In [33]:
max_city_zips = normalized_zip_city_counts[
    normalized_zip_city_counts == 4
].index

geolocation[
    geolocation["geolocation_zip_code_prefix"].isin(
        max_city_zips
    )
][
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "city_normalized",
        "geolocation_state"
    ]
].drop_duplicates().sort_values(
    [
        "geolocation_zip_code_prefix",
        "city_normalized"
    ]
)

,geolocation_zip_code_prefix,geolocation_city,city_normalized,geolocation_state
204960,13454,santa barbara d oeste,santa barbara d oeste,SP
204649,13454,santa barbara d'oeste,santa barbara d'oeste,SP
204702,13454,santa bárbara d'oeste,santa barbara d'oeste,SP
207792,13454,santa bárbara d`oeste,santa barbara d`oeste,SP
205228,13454,santa bárbara doeste,santa barbara doeste,SP
263457,17970,sao joao do pau d alho,sao joao do pau d alho,SP
263700,17970,são joão do pau d%26apos%3balho,sao joao do pau d%26apos%3balho,SP
263330,17970,sao joao do pau d'alho,sao joao do pau d'alho,SP
263426,17970,são joão do pau d'alho,sao joao do pau d'alho,SP
263598,17970,sao joao do pau dalho,sao joao do pau dalho,SP


In [34]:
encoding_artifacts = geolocation[
    geolocation["geolocation_city"].str.contains(
        r"%[0-9A-Fa-f]{2}",
        regex=True
    )
]

print(
    "Rows with encoded artifacts:",
    len(encoding_artifacts)
)

print(
    "Unique city values with encoded artifacts:",
    encoding_artifacts["geolocation_city"].nunique()
)

print(
    encoding_artifacts["geolocation_city"]
    .value_counts()
)

Rows with encoded artifacts: 2
Unique city values with encoded artifacts: 2
geolocation_city
são joão do pau d%26apos%3balho    1
lambari d%26apos%3boeste           1
Name: count, dtype: int64


In [35]:
geolocation["geolocation_city"] = (
    geolocation["geolocation_city"]
    .str.replace(
        "%26apos%3b",
        "'",
        regex=False
    )
)

print(
    "Rows with encoded artifacts:",
    geolocation["geolocation_city"]
    .str.contains(r"%[0-9A-Fa-f]{2}", regex=True)
    .sum()
)

print(
    geolocation.loc[
        geolocation["geolocation_zip_code_prefix"].isin(
            [17970]
        ),
        "geolocation_city"
    ]
    .drop_duplicates()
    .sort_values()
)

Rows with encoded artifacts: 0
263457    sao joao do pau d alho
263330    sao joao do pau d'alho
263598     sao joao do pau dalho
263426    são joão do pau d'alho
Name: geolocation_city, dtype: str


In [36]:
geolocation["city_normalized"] = (
    geolocation["geolocation_city"]
    .apply(normalize_city)
)

normalized_zip_city_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["city_normalized"]
    .nunique()
)

print(
    "ZIP prefixes with multiple normalized cities:",
    (normalized_zip_city_counts > 1).sum()
)

print(
    "Maximum normalized cities for one ZIP prefix:",
    normalized_zip_city_counts.max()
)

ZIP prefixes with multiple normalized cities: 544
Maximum normalized cities for one ZIP prefix: 4


In [37]:
geolocation = geolocation.drop(
    columns="city_normalized"
)

print(geolocation.columns.tolist())
print("Shape:", geolocation.shape)

['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
Shape: (738299, 5)


In [38]:
print("Final shape:", geolocation.shape)

print("\nMissing values:")
print(geolocation.isna().sum())

print(
    "\nDuplicate rows:",
    geolocation.duplicated().sum()
)

print("\nData types:")
print(geolocation.dtypes)

print(
    "\nInvalid latitude values:",
    (~geolocation["geolocation_lat"].between(-90, 90)).sum()
)

print(
    "Invalid longitude values:",
    (~geolocation["geolocation_lng"].between(-180, 180)).sum()
)

print(
    "Potential coordinate outliers:",
    (
        ~(
            geolocation["geolocation_lat"].between(-35, 6)
            &
            geolocation["geolocation_lng"].between(-75, -30)
        )
    ).sum()
)

print(
    "ZIP prefixes with multiple states:",
    (
        geolocation
        .groupby("geolocation_zip_code_prefix")["geolocation_state"]
        .nunique()
        .gt(1)
        .sum()
    )
)

print(
    "City spacing issues:",
    (
        geolocation["geolocation_city"]
        .ne(geolocation["geolocation_city"].str.strip())
        |
        geolocation["geolocation_city"]
        .str.contains(r"\s{2,}", regex=True)
    ).sum()
)

print(
    "Encoded city artifacts:",
    geolocation["geolocation_city"]
    .str.contains(r"%[0-9A-Fa-f]{2}", regex=True)
    .sum()
)

Final shape: (738299, 5)

Missing values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Duplicate rows: 0

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Invalid latitude values: 0
Invalid longitude values: 0
Potential coordinate outliers: 0
ZIP prefixes with multiple states: 0
City spacing issues: 0
Encoded city artifacts: 0


In [39]:
output_path = "../CYRV_geolocation_cleaned.csv"

geolocation.to_csv(
    output_path,
    index=False
)

print("Saved to:", output_path)

Saved to: ../CYRV_geolocation_cleaned.csv


In [40]:
saved_geolocation = pd.read_csv(
    "../CYRV_geolocation_cleaned.csv"
)

print("Saved shape:", saved_geolocation.shape)

print("\nMissing values:")
print(saved_geolocation.isna().sum())

print(
    "\nDuplicate rows:",
    saved_geolocation.duplicated().sum()
)

print("\nData types:")
print(saved_geolocation.dtypes)

Saved shape: (738299, 5)

Missing values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Duplicate rows: 0

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object
